# 04 RL Training (DDQN)
Training the agent for risk state classification.

In [ ]:
import pandas as pd
import numpy as np
from src.ddqn_agent import DDQNAgent
from src.reward_function import calculate_rho
import torch

X = pd.read_csv('../data/processed/selected_features.csv', index_col=0, parse_dates=True)
modeling = pd.read_csv('../data/processed/modeling_results.csv', index_col=0, parse_dates=True)
dataset = X.join(modeling[['vol_GARCH', 'vol_GJR', 'risk_label']], how='inner')
X_data = dataset.drop(columns=['risk_label'])
y_data = dataset['risk_label']

# Chronological Splitting
train_idx = dataset.index <= '2018-12-31'
val_idx = (dataset.index > '2018-12-31') & (dataset.index <= '2022-12-31')
test_idx = dataset.index > '2022-12-31'

X_train, y_train = X_data[train_idx], y_data[train_idx]
rho = calculate_rho(y_train)

agent = DDQNAgent(X_train.shape[1], 2, rho)
EPISODES = 20

print("Training RL Agent...")
for e in range(EPISODES):
    total_reward = 0
    for i in range(len(X_train) - 1):
        state = X_train.iloc[i].values
        action = agent.act(state)
        reward = agent.calculate_reward(action, y_train.iloc[i])
        next_state = X_train.iloc[i+1].values
        done = (i == len(X_train) - 2)
        agent.remember(state, action, reward, next_state, done)
        total_reward += reward
        if i % 20 == 0: agent.replay(64)
    agent.update_target_model()
    print(f"Episode {e+1}/{EPISODES} | Reward: {total_reward:.2f} | Epsilon: {agent.epsilon:.2f}")

agent.save('../models/saved_models/ddqn_agent.pth')